# 06 — Diversity & Inclusion
**Goal**: Measure and visualize DEI metrics across the organization

**ML Progression**: Statistical → Chi-square → Simpson Index → Intersectional Analysis

**HR Value**: DEI compliance, targeted hiring strategies

**Employee Value**: Inclusive workplace awareness

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
from pathlib import Path
from scipy.stats import chi2_contingency

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

cwd = Path.cwd()
if (cwd / 'data/raw/employee_data.csv').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'data/raw/employee_data.csv').exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError('Cannot find project root')
os.chdir(PROJECT_ROOT)
PROJECT_ROOT = Path.cwd().resolve()

ANALYSIS_DIR = PROJECT_ROOT / 'data/analysis/06_diversity'
FIGURES_DIR = PROJECT_ROOT / 'reports/figures'
Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(ANALYSIS_DIR / 'dataset.parquet')
print(f'Loaded: {len(df)} employees, {len(df.columns)} cols')

## 1. Demographic Overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df['gender_code'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=axes[0], title='Gender')
df['race_desc'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=axes[1], title='Race')
df['marital_desc'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=axes[2], title='Marital Status')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/06_demographics.png', bbox_inches='tight')
plt.show()

## 2. Chi-Square: Gender × Department

In [ ]:
ct = pd.crosstab(df['gender_code'], df['department_type'])
chi2, p, dof, expected = chi2_contingency(ct)

print(f'Chi-Square: {chi2:.2f}')
print(f'p-value: {p:.4f}')
print(f'Degrees of freedom: {dof}')
print(f'\nInterpretation: {"Significant" if p < 0.05 else "Not significant"} association between gender and department')

# Cramer's V
n = ct.sum().sum()
v = np.sqrt(chi2 / (n * min(ct.shape) - 1))
print(f"Cramer's V: {v:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
pd.crosstab(df['department_type'], df['gender_code'], normalize='index').plot(
    kind='bar', stacked=True, ax=ax, title='Gender Distribution by Department')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/06_gender_by_dept.png', bbox_inches='tight')
plt.show()

## 3. Simpson Diversity Index by Department

In [ ]:
def simpson_index(series):
    """1 - sum(pi^2), where pi = proportion of each category."""
    counts = series.value_counts()
    n = counts.sum()
    return 1 - sum((c / n) ** 2 for c in counts)

dept_diversity = df.groupby('department_type').agg({
    'gender_code': simpson_index,
    'race_desc': simpson_index,
}).round(3)
dept_diversity.columns = ['Gender Diversity', 'Race Diversity']
print('Diversity by Department:')
print(dept_diversity)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
dept_diversity.plot(kind='bar', ax=ax, title='Simpson Diversity Index by Department')
ax.set_ylabel('Diversity (1 = most diverse)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/06_diversity_index.png', bbox_inches='tight')
plt.show()

## 4. Intersectional Heatmap

In [ ]:
ct_intersection = pd.crosstab([df['gender_code'], df['race_desc']], df['department_type'])
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(ct_intersection, annot=True, fmt='d', cmap='YlOrRd', ax=ax)
ax.set_title('Gender × Race × Department')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/06_intersectional_heatmap.png', bbox_inches='tight')
plt.show()

## 5. Key Takeaways

In [ ]:
print('--- Key Insights ---')
print(f'1. Gender-department association: p={p:.4f} ({"significant" if p < 0.05 else "not significant"})')
print(f'2. Most diverse department: {dept_diversity["Gender Diversity"].idxmax()}')
print(f'3. Least diverse department: {dept_diversity["Gender Diversity"].idxmin()}')
print()
print('--- HR Action Items ---')
print('- Focus recruitment on under-represented groups in low-diversity departments')
print('- Set DEI targets at department level')
print()
print('--- Employee Impact ---')
print('- Transparent diversity metrics build inclusive culture')
print('- Intersectional analysis highlights compound representation gaps')